In [ ]:
# ------------------------------------------------------------------------------
# VULNERABILITY DATA PIPELINE
# Complete data preprocessing and feature engineering for vulnerability prediction
# ------------------------------------------------------------------------------
# This notebook processes vulnerability scan data and web reconnaissance data
# to create a clean, feature-rich dataset for machine learning training.
# ------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import os
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('ggplot')
sns.set_palette("husl")

print("=" * 80)
print("VULNERABILITY DATA PIPELINE - INITIALIZED")
print("=" * 80)
print(f"Working directory: {os.getcwd()}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print("=" * 80)

In [2]:
# ------------------------------------------------------------------------------
# 1. LOAD DATASETS
# ------------------------------------------------------------------------------

# Define file paths
BASE_PATH = r'C:\D\SMART-HUNTER\Data'
VULN_PATH = os.path.join(BASE_PATH, 'Datasets', 'vulnerability_ml_dataset.csv')
RECON_PATH = os.path.join(BASE_PATH, 'Datasets', 'web_recon_ml_dataset.csv')

# Load datasets
print("\n📊 LOADING DATASETS...")
print("-" * 50)

df_vuln = pd.read_csv(VULN_PATH, low_memory=False)
df_recon = pd.read_csv(RECON_PATH, low_memory=False)

print(f"Vulnerability dataset: {df_vuln.shape[0]} rows, {df_vuln.shape[1]} columns")
print(f"Reconnaissance dataset: {df_recon.shape[0]} rows, {df_recon.shape[1]} columns")

# Display basic info
print("\n📋 Vulnerability dataset columns (first 10):")
print(df_vuln.columns[:10].tolist())

print("\n📋 Reconnaissance dataset columns (first 10):")
print(df_recon.columns[:10].tolist())


📊 LOADING DATASETS...
--------------------------------------------------
Vulnerability dataset: 228 rows, 49 columns
Reconnaissance dataset: 1002 rows, 109 columns

📋 Vulnerability dataset columns (first 10):
['scan_id', 'url', 'domain', 'timestamp', 'total_vulnerabilities', 'has_sql_injection', 'has_xss', 'has_command_injection', 'has_path_traversal', 'has_file_inclusion']

📋 Reconnaissance dataset columns (first 10):
['url_length', 'has_https', 'path_depth', 'has_query_params', 'num_query_params', 'has_fragment', 'has_port', 'subdomain_count', 'is_ip_address', 'domain_tld']


In [3]:
# ------------------------------------------------------------------------------
# 2. ANALYZE IDs AND EXTRACT DOMAINS FOR MERGING
# ------------------------------------------------------------------------------

print("\n🔍 ANALYZING IDs AND EXTRACTING DOMAINS...")
print("-" * 50)

# Function to extract main domain from URL
def extract_main_domain(url):
    """
    Extract the main domain (example.com) from a full URL.
    
    Args:
        url: Full URL string
    
    Returns:
        Main domain or 'unknown' if extraction fails
    """
    try:
        url = str(url).lower()
        # Remove protocol
        url = re.sub(r'https?://(www\.)?', '', url)
        # Take first part before slash
        url = url.split('/')[0]
        # Extract main domain (example.com from sub.example.com)
        parts = url.split('.')
        if len(parts) >= 2:
            return '.'.join(parts[-2:])  # example.com
        return url
    except:
        return 'unknown'

# Extract domains from vulnerability dataset
df_vuln['main_domain'] = df_vuln['url'].apply(extract_main_domain)

# Extract domains from reconnaissance dataset (try different URL columns)
if 'target_url' in df_recon.columns:
    df_recon['main_domain'] = df_recon['target_url'].apply(extract_main_domain)
elif 'original_url' in df_recon.columns:
    df_recon['main_domain'] = df_recon['original_url'].apply(extract_main_domain)
elif 'url' in df_recon.columns:
    df_recon['main_domain'] = df_recon['url'].apply(extract_main_domain)

# Analyze scan_id matches
vuln_ids = set(df_vuln['scan_id'].astype(str).unique())
recon_ids = set(df_recon['scan_id'].astype(str).unique())
common_ids = vuln_ids & recon_ids

print(f"Vulnerability dataset - unique scan_ids: {len(vuln_ids)}")
print(f"Reconnaissance dataset - unique scan_ids: {len(recon_ids)}")
print(f"Common scan_ids: {len(common_ids)}")

# Analyze domain matches
vuln_domains = set(df_vuln['main_domain'].unique())
recon_domains = set(df_recon['main_domain'].unique())
common_domains = vuln_domains & recon_domains

print(f"\nVulnerability dataset - unique domains: {len(vuln_domains)}")
print(f"Reconnaissance dataset - unique domains: {len(recon_domains)}")
print(f"Common domains: {len(common_domains)}")

if len(common_domains) > 0:
    print(f"Sample common domains: {list(common_domains)[:5]}")


🔍 ANALYZING IDs AND EXTRACTING DOMAINS...
--------------------------------------------------
Vulnerability dataset - unique scan_ids: 227
Reconnaissance dataset - unique scan_ids: 211
Common scan_ids: 2

Vulnerability dataset - unique domains: 195
Reconnaissance dataset - unique domains: 202
Common domains: 187
Sample common domains: ['tron.network', 'temu.com', 'helium.com', 'zerobounce.net', 'wisdomtree.com']


In [4]:
# ------------------------------------------------------------------------------
# 3. DATA CLEANING AND PREPROCESSING
# ------------------------------------------------------------------------------

print("\n🧹 CLEANING DATASETS...")
print("-" * 50)

def clean_dataframe(df, name):
    """
    Clean a dataframe by handling missing values and standardizing strings.
    
    Args:
        df: Input dataframe
        name: Dataset name for logging
    
    Returns:
        Cleaned dataframe
    """
    df_clean = df.copy()
    
    # Standardize column names
    df_clean.columns = [col.strip().replace(' ', '_').lower() for col in df_clean.columns]
    
    # Handle numeric columns
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    df_clean[numeric_cols] = df_clean[numeric_cols].fillna(0)
    
    # Handle string columns
    string_cols = df_clean.select_dtypes(include=['object']).columns
    for col in string_cols:
        df_clean[col] = df_clean[col].astype(str).fillna('unknown').str.lower()
    
    print(f"  ✓ {name}: {len(df_clean)} rows, {len(df_clean.columns)} columns")
    return df_clean

# Clean both datasets
df_vuln_clean = clean_dataframe(df_vuln, "Vulnerability dataset")
df_recon_clean = clean_dataframe(df_recon, "Reconnaissance dataset")


🧹 CLEANING DATASETS...
--------------------------------------------------
  ✓ Vulnerability dataset: 228 rows, 50 columns
  ✓ Reconnaissance dataset: 1002 rows, 110 columns


In [5]:
# ------------------------------------------------------------------------------
# 4. MULTIPLE MERGING STRATEGIES
# ------------------------------------------------------------------------------

print("\n🔄 MERGING DATASETS...")
print("-" * 50)

# Strategy 1: Merge by scan_id
merged_by_id = pd.merge(
    df_recon_clean, df_vuln_clean,
    on='scan_id',
    how='inner',
    suffixes=('', '_vuln')
)
print(f"Strategy 1 (scan_id merge): {len(merged_by_id)} rows")

# Strategy 2: Merge by domain
merged_by_domain = pd.merge(
    df_recon_clean, df_vuln_clean,
    on='main_domain',
    how='inner',
    suffixes=('', '_vuln')
)
print(f"Strategy 2 (domain merge): {len(merged_by_domain)} rows")

# Strategy 3: Outer merge (keeps all records)
merged_outer = pd.merge(
    df_recon_clean, df_vuln_clean,
    on='main_domain',
    how='outer',
    suffixes=('_recon', '_vuln')
)
print(f"Strategy 3 (outer merge): {len(merged_outer)} rows")

# Select the best merging strategy
if len(merged_by_id) >= len(merged_by_domain) and len(merged_by_id) > 0:
    df_merged = merged_by_id
    merge_method = "scan_id"
elif len(merged_by_domain) > 0:
    df_merged = merged_by_domain
    merge_method = "domain"
else:
    df_merged = merged_outer
    merge_method = "outer"

print(f"\n✅ Selected merge method: {merge_method}")
print(f"✅ Final merged dataset: {len(df_merged)} rows, {len(df_merged.columns)} columns")

# Clean up duplicate columns
vuln_dup_cols = [col for col in df_merged.columns if col.endswith('_vuln')]
recon_dup_cols = [col for col in df_merged.columns if col.endswith('_recon')]
df_merged.drop(columns=vuln_dup_cols + recon_dup_cols, inplace=True, errors='ignore')

# Combine URL columns if they exist
if 'url_recon' in df_merged.columns and 'url_vuln' in df_merged.columns:
    df_merged['url'] = df_merged['url_recon'].fillna(df_merged['url_vuln'])
    df_merged.drop(columns=['url_recon', 'url_vuln'], inplace=True)

print(f"\n📊 Final shape: {df_merged.shape}")


🔄 MERGING DATASETS...
--------------------------------------------------
Strategy 1 (scan_id merge): 2 rows
Strategy 2 (domain merge): 687 rows
Strategy 3 (outer merge): 1077 rows

✅ Selected merge method: domain
✅ Final merged dataset: 687 rows, 159 columns

📊 Final shape: (687, 151)


In [6]:
# ------------------------------------------------------------------------------
# 5. FEATURE ENGINEERING
# ------------------------------------------------------------------------------

print("\n⚙️ FEATURE ENGINEERING...")
print("-" * 50)

df = df_merged.copy()
initial_cols = len(df.columns)

# Ensure vulnerability columns exist
vuln_columns = ['total_vulnerabilities', 'has_sql_injection', 'has_xss', 
                'has_command_injection', 'critical_vuln_count', 'high_vuln_count']

for col in vuln_columns:
    if col not in df.columns:
        df[col] = 0
    else:
        df[col] = df[col].fillna(0)

# Target variable
df['has_any_vulnerability'] = (df['total_vulnerabilities'] > 0).astype(int)

# Risk features
df['critical_high_ratio'] = df.apply(
    lambda x: (x['critical_vuln_count'] + x['high_vuln_count']) / max(x['total_vulnerabilities'], 1), 
    axis=1
)

# Interactivity features
has_forms = df.get('has_forms', pd.Series(0, index=df.index))
has_inputs = df.get('has_inputs', pd.Series(0, index=df.index))
has_javascript = df.get('has_javascript', pd.Series(0, index=df.index))
form_count = df.get('form_count', pd.Series(0, index=df.index))
input_count = df.get('input_count', pd.Series(0, index=df.index))

df['interactivity_score'] = (
    has_forms * 0.3 + 
    has_inputs * 0.3 + 
    has_javascript * 0.2 +
    (form_count / (form_count + 1)) * 0.1 +
    (input_count / (input_count + 1)) * 0.1
)

# Security features
security_cols = ['has_https', 'has_csp', 'has_hsts', 'has_frame_options', 
                 'has_xss_protection', 'has_content_type_options']

df['security_score'] = 0
for col in security_cols:
    if col in df.columns:
        df['security_score'] += df[col] * 0.2

if 'security_headers_count' in df.columns:
    df['security_score'] += (df['security_headers_count'] / 10) * 0.2

# WAF detection
waf_cols = [col for col in df.columns if 'waf_' in col]
if waf_cols:
    df['waf_detected'] = df[waf_cols].any(axis=1).astype(int)

# Size-based features
if 'response_size' in df.columns:
    df['log_response_size'] = np.log1p(df['response_size'])
    df['response_size_kb'] = df['response_size'] / 1024

if 'response_time_ms' in df.columns:
    df['response_time_sec'] = df['response_time_ms'] / 1000

# Parameter features
if 'num_query_params' in df.columns and 'input_count' in df.columns:
    df['params_per_input'] = df.apply(
        lambda x: x['num_query_params'] / max(x['input_count'], 1), 
        axis=1
    )

# Density features
if 'total_vulnerabilities' in df.columns and 'response_size' in df.columns:
    df['vuln_density_per_kb'] = df.apply(
        lambda x: x['total_vulnerabilities'] / max(x['response_size'] / 1024, 0.001), 
        axis=1
    )

new_features_count = len(df.columns) - initial_cols
print(f"✅ Created {new_features_count} new features")
print(f"✅ Total features now: {len(df.columns)}")


⚙️ FEATURE ENGINEERING...
--------------------------------------------------
✅ Created 8 new features
✅ Total features now: 159


In [7]:
# ------------------------------------------------------------------------------
# 6. CATEGORICAL ENCODING
# ------------------------------------------------------------------------------

print("\n🔢 ENCODING CATEGORICAL FEATURES...")
print("-" * 50)

# -- DEFENSIVE CHECK FOR df --
if "df" not in locals():
    if "df_merged" in locals():
        df = df_merged.copy()
        print("⚠️ Warning: 'df' was not defined. Initialized from 'df_merged'.")
    else:
        print("❌ Error: 'df' is not defined. Please run Section 5 first.")
        # Define an empty/dummy df if missing to avoid NameError crash but let user know
        raise NameError("name 'df' is not defined. You MUST run Section 5 (Feature Engineering) before this cell.")

label_encoders = {}
categorical_cols = ['domain_tld', 'server_header', 'content_type', 'status_category']

encoded_count = 0
for col in categorical_cols:
    if col in df.columns:
        # Fill missing values
        df[col] = df[col].astype(str).fillna('unknown')
        
        # Apply label encoding
        le = LabelEncoder()
        df[f"{col}_encoded"] = le.fit_transform(df[col])
        label_encoders[col] = le
        encoded_count += 1
        
        print(f"  ✓ Encoded {col}: {df[col].nunique()} unique values")

print(f"✅ Encoded {encoded_count} categorical columns")


🔢 ENCODING CATEGORICAL FEATURES...
--------------------------------------------------
  ✓ Encoded domain_tld: 26 unique values
  ✓ Encoded server_header: 31 unique values
  ✓ Encoded content_type: 4 unique values
  ✓ Encoded status_category: 3 unique values
✅ Encoded 4 categorical columns


In [8]:
# ------------------------------------------------------------------------------
# 7. FEATURE NORMALIZATION
# ------------------------------------------------------------------------------

print("\n📏 NORMALIZING NUMERICAL FEATURES...")
print("-" * 50)

# Identify columns to normalize (exclude binary and target columns)
exclude_patterns = ['has_', 'is_', 'encoded', 'waf_', 'score', 'ratio', 
                    'vulnerability', 'scan_id', 'url', 'domain']

numeric_cols = []
for col in df.select_dtypes(include=[np.number]).columns:
    if not any(pattern in col for pattern in exclude_patterns):
        if df[col].nunique() > 2:  # Not binary
            numeric_cols.append(col)

print(f"Found {len(numeric_cols)} numerical columns for normalization")

if numeric_cols:
    # Min-Max scaling
    scaler = MinMaxScaler()
    df_numeric = df[numeric_cols].fillna(0)
    df_scaled = scaler.fit_transform(df_numeric)
    
    for i, col in enumerate(numeric_cols):
        df[f'{col}_scaled'] = df_scaled[:, i]
    
    print(f"✅ Normalized {len(numeric_cols)} columns using MinMaxScaler")
    
    # Show sample statistics
    print("\n📊 Normalization statistics (sample):")
    sample_cols = numeric_cols[:3]
    for col in sample_cols:
        print(f"  {col}: min={df[f'{col}_scaled'].min():.3f}, "
              f"max={df[f'{col}_scaled'].max():.3f}, "
              f"mean={df[f'{col}_scaled'].mean():.3f}")


📏 NORMALIZING NUMERICAL FEATURES...
--------------------------------------------------
Found 48 numerical columns for normalization
✅ Normalized 48 columns using MinMaxScaler

📊 Normalization statistics (sample):
  path_depth: min=0.000, max=1.000, mean=0.207
  num_query_params: min=0.000, max=1.000, mean=0.014
  status_code: min=0.000, max=1.000, mean=0.140


In [9]:
# ------------------------------------------------------------------------------
# 8. PREPARE FINAL TRAINING DATASET
# ------------------------------------------------------------------------------

print("\n🎯 PREPARING FINAL TRAINING DATASET...")
print("-" * 50)

# Define feature groups
feature_groups = {
    'url_characteristics': [
        'url_length', 'path_depth', 'num_query_params', 'subdomain_count',
        'has_https', 'url_length_scaled', 'path_depth_scaled'
    ],
    
    'security_headers': [
        'security_headers_count', 'has_csp', 'has_hsts', 'has_frame_options',
        'has_xss_protection', 'security_headers_count_scaled'
    ],
    
    'content_features': [
        'response_size', 'response_time_ms', 'total_headers',
        'response_size_scaled', 'response_time_ms_scaled', 'log_response_size'
    ],
    
    'interaction_features': [
        'has_forms', 'has_inputs', 'has_javascript', 'has_comments',
        'form_count', 'input_count', 'interactivity_score',
        'form_count_scaled', 'input_count_scaled'
    ],
    
    'waf_detection': [
        'waf_cloudflare_encoded', 'waf_aws_encoded', 'waf_imperva_encoded',
        'waf_detected' if 'waf_detected' in df.columns else None
    ],
    
    'technology_stack': [
        'tech_php_encoded', 'tech_aspnet_encoded', 'tech_wordpress_encoded',
        'tech_drupal_encoded', 'tech_joomla_encoded'
    ],
    
    'engineered_features': [
        'critical_high_ratio', 'security_score', 'vuln_density_per_kb',
        'params_per_input' if 'params_per_input' in df.columns else None
    ],
    
    'encoded_categorical': [
        'domain_tld_encoded', 'server_header_encoded', 
        'content_type_encoded', 'status_category_encoded'
    ]
}

# Compile all features
all_features = []
for group_name, features in feature_groups.items():
    valid_features = [f for f in features if f and f in df.columns]
    all_features.extend(valid_features)
    if valid_features:
        print(f"  ✓ {group_name}: {len(valid_features)} features")

# Remove duplicates
all_features = list(set(all_features))
print(f"\n✅ Total selected features: {len(all_features)}")

# Define target
target = 'has_any_vulnerability'

# Create training dataframe
available_cols = all_features + [target]
available_cols = [col for col in available_cols if col in df.columns]

training_df = df[available_cols].copy()

# Remove rows with NaN
training_df = training_df.dropna()

# Add identifiers for reference
id_columns = []
for id_col in ['scan_id', 'url', 'main_domain']:
    if id_col in df.columns:
        training_df[id_col] = df.loc[training_df.index, id_col].values
        id_columns.append(id_col)

print(f"\n📊 Final training dataset:")
print(f"  - Samples: {len(training_df)}")
print(f"  - Features: {len(all_features)}")
print(f"  - Identifiers: {', '.join(id_columns) if id_columns else 'None'}")

# Show target distribution
if target in training_df.columns:
    class_counts = training_df[target].value_counts()
    class_percentages = training_df[target].value_counts(normalize=True) * 100
    
    print(f"\n📈 Target distribution:")
    for val in sorted(class_counts.index):
        print(f"  - Class {val}: {class_counts[val]} samples ({class_percentages[val]:.1f}%)")


🎯 PREPARING FINAL TRAINING DATASET...
--------------------------------------------------
  ✓ url_characteristics: 6 features
  ✓ security_headers: 6 features
  ✓ content_features: 6 features
  ✓ interaction_features: 9 features
  ✓ waf_detection: 1 features
  ✓ engineered_features: 4 features
  ✓ encoded_categorical: 4 features

✅ Total selected features: 36

📊 Final training dataset:
  - Samples: 687
  - Features: 36
  - Identifiers: scan_id, url, main_domain

📈 Target distribution:
  - Class 0: 75 samples (10.9%)
  - Class 1: 612 samples (89.1%)


In [5]:
# ------------------------------------------------------------------------------
# 9. DATA VISUALIZATION
# ------------------------------------------------------------------------------

print("\n📊 GENERATING VISUALIZATIONS...")
print("-" * 50)

import matplotlib.pyplot as plt
import seaborn as sns

# -- DEFENSIVE CHECK FOR training_df and plt --
if "training_df" not in locals():
    print("⚠️ Warning: 'training_df' is not defined. Attempting fallback if possible...")
    if "df" in locals():
        target = "has_any_vulnerability"
        if target in df.columns:
            training_df = df.copy()
    else:
        print("❌ Error: Variable 'training_df' missing. Please run Section 8 first.")
        raise NameError("training_df missing")

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Target distribution
ax1 = axes[0, 0]
target = "has_any_vulnerability"
if target in training_df.columns:
    colors = ['#ff6b6b', '#4ecdc4']
    counts = training_df[target].value_counts()
    ax1.bar(['Clean', 'Vulnerable'][:len(counts)], counts.values, color=colors[:len(counts)], edgecolor='black', linewidth=1.5)
    ax1.set_title('Target Distribution: Vulnerable vs Clean Sites', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Number of Sites')
    for i, v in enumerate(counts.values):
        ax1.text(i, v + 5, str(v), ha='center', fontweight='bold')

# 2. Feature correlation with target
ax2 = axes[0, 1]
if target in training_df.columns:
    # Calculate correlation with target
    numeric_cols = training_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        correlations = training_df[numeric_cols].corr()[target].sort_values(ascending=False)
        top_correlations = correlations[1:11]  # Top 10 (excluding self-correlation)
        
        colors = ['#4ecdc4' if c > 0 else '#ff6b6b' for c in top_correlations.values]
        ax2.barh(range(len(top_correlations)), top_correlations.values, color=colors)
        ax2.set_yticks(range(len(top_correlations)))
        ax2.set_yticklabels(top_correlations.index, fontsize=10)
        ax2.set_title('Top 10 Features Correlated with Vulnerabilities', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Correlation Coefficient')
        ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# 3. Vulnerability count distribution
ax3 = axes[1, 0]
if "df" in locals() and 'total_vulnerabilities' in df.columns:
    vuln_counts = df[df['total_vulnerabilities'] > 0]['total_vulnerabilities']
    if len(vuln_counts) > 0:
        ax3.hist(vuln_counts, bins=20, color='#ff6b6b', edgecolor='black', alpha=0.7)
        ax3.set_title('Distribution of Vulnerability Counts', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Number of Vulnerabilities')
        ax3.set_ylabel('Frequency')

# 4. Security score vs vulnerability
ax4 = axes[1, 1]
if 'security_score' in training_df.columns and target in training_df.columns:
    clean_scores = training_df[training_df[target] == 0]['security_score']
    vuln_scores = training_df[training_df[target] == 1]['security_score']
    
    if len(clean_scores) > 0 and len(vuln_scores) > 0:
        ax4.boxplot([clean_scores, vuln_scores], labels=['Clean Sites', 'Vulnerable Sites'])
        ax4.set_title('Security Score Distribution', fontsize=14, fontweight='bold')
        ax4.set_ylabel('Security Score')

plt.tight_layout()
plt.savefig('data_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualization saved as 'data_visualization.png'")


📊 GENERATING VISUALIZATIONS...
--------------------------------------------------


NameError: name 'plt' is not defined

In [4]:
# ------------------------------------------------------------------------------
# 10. SAVE RESULTS
# ------------------------------------------------------------------------------

print("\n💾 SAVING RESULTS...")
print("-" * 50)

# -- DEFENSIVE CHECK FOR training_df --
if "training_df" in locals() and len(training_df) > 0:
    if "BASE_PATH" not in locals():
        BASE_PATH = r"C:\D\SMART-HUNTER\Data"
        
    output_file = os.path.join(BASE_PATH, 'final_training_data.csv')
    try:
        training_df.to_csv(output_file, index=False)
        print(f"✅ Successfully saved training data to {output_file}")
        print(f"   Total samples: {len(training_df)}")
        print(f"   Total features: {len(training_df.columns)}")
    except Exception as e:
        print(f"❌ Error saving dataset: {e}")
else:
    print("⚠️ No data to save. Please run Section 8 (Prepare Final Dataset) first.")


💾 SAVING RESULTS...
--------------------------------------------------


NameError: name 'training_df' is not defined

In [3]:
# ------------------------------------------------------------------------------
# 11. TRAIN-TEST SPLIT (Optional - for immediate modeling)
# ------------------------------------------------------------------------------

print("\n🔄 PREPARING TRAIN-TEST SPLIT...")
print("-" * 50)

# -- DEFENSIVE CHECK FOR training_df --
if "training_df" not in locals():
    print("❌ Error: 'training_df' is not defined. Please run Section 8 first.")
else:
    if len(training_df) > 10:  # Only if we have enough samples
        # Separate features and target
        feature_cols = [col for col in training_df.columns 
                       if col not in [target] + id_columns]
        
        X = training_df[feature_cols]
        y = training_df[target]
        
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        print(f"Training set: {X_train.shape[0]} samples")
        print(f"Test set: {X_test.shape[0]} samples")
        print(f"Training target distribution:\n  {y_train.value_counts(normalize=True).to_dict()}")
        
        # Save split datasets
        train_file = os.path.join(BASE_PATH, 'X_train.csv')
        X_train.to_csv(train_file, index=False)
        
        test_file = os.path.join(BASE_PATH, 'X_test.csv')
        X_test.to_csv(test_file, index=False)
        
        y_train.to_csv(os.path.join(BASE_PATH, 'y_train.csv'), index=False)
        y_test.to_csv(os.path.join(BASE_PATH, 'y_test.csv'), index=False)
        
        print(f"\n✅ Train-test split saved")
    else:
        print("⚠️ Not enough samples for train-test split")


🔄 PREPARING TRAIN-TEST SPLIT...
--------------------------------------------------


NameError: name 'training_df' is not defined